# Orbit Wars 게임 시각화

현재 학습된 모델 (`notebooks/model.pt`) 로 self-play 게임 1판 실행 + Kaggle 인터랙티브 플레이어로 재생.

**사용법**
1. Jupyter 에서 셀 위에서부터 차례로 실행
2. 마지막 셀의 플레이어로 ▶/⏸ 재생, ◀/▶ 프레임 스텝, ⏩ 속도 조정 가능

**모델 교체** — `ORBIT_WEIGHTS` 환경변수로 다른 .pt 지정 가능 (셀 1):
```python
os.environ['ORBIT_WEIGHTS'] = '/path/to/other_model.pt'
```

## 1. 환경 / agent 로드

In [ ]:
import os, sys
from pathlib import Path

# repo root 와 submission/ 둘 다 sys.path 에 추가 (notebook 실행 위치 무관 동작).
_HERE = Path.cwd().resolve()
REPO  = _HERE.parent if _HERE.name == 'notebooks' else _HERE
for p in (str(REPO), str(REPO / 'submission')):
    if p not in sys.path:
        sys.path.insert(0, p)

# 모델 경로 — ORBIT_WEIGHTS env var 가 우선, 없으면 notebooks/model.pt.
os.environ.setdefault('ORBIT_WEIGHTS', str(REPO / 'notebooks' / 'model.pt'))
print('ORBIT_WEIGHTS:', os.environ['ORBIT_WEIGHTS'])
print('exists:', Path(os.environ['ORBIT_WEIGHTS']).exists())

In [ ]:
from kaggle_environments import make
from main import agent  # submission/main.py — train.py 의 analyze/decode 그대로 사용

env = make('orbit_wars', debug=False)
print('env ready:', env.specification['name'])

## 2. 2-player self-play 게임 실행

양쪽 모두 같은 모델. 첫 호출이 모델 로드 포함이라 약간 느림.

In [ ]:
import time
t0 = time.time()
env.run([agent, agent])
dt = time.time() - t0

p0_reward = env.state[0].reward
p1_reward = env.state[1].reward
winner = 'P0' if p0_reward == 1 else ('P1' if p1_reward == 1 else 'DRAW')

print(f'duration:   {dt:.2f}s')
print(f'steps:      {len(env.steps)}')
print(f'p0 reward:  {p0_reward}')
print(f'p1 reward:  {p1_reward}')
print(f'winner:     {winner}')
print(f'statuses:   {[s.status for s in env.state]}')

## 3. 종료 시점 행성 / fleet 분포

In [ ]:
from collections import Counter
obs = env.state[0].observation
planets = obs.get('planets', []) if isinstance(obs, dict) else obs.planets
fleets  = obs.get('fleets',  []) if isinstance(obs, dict) else obs.fleets

owner_count = Counter()
ships_by_owner = Counter()
prod_by_owner  = Counter()
for p in planets:
    owner = p[1] if isinstance(p, (list, tuple)) else p.owner
    ships = p[5] if isinstance(p, (list, tuple)) else p.ships
    prod  = p[6] if isinstance(p, (list, tuple)) else p.production
    owner_count[owner] += 1
    ships_by_owner[owner] += ships
    prod_by_owner[owner]  += prod

fleet_ships_by_owner = Counter()
for f in fleets:
    o = f[1] if isinstance(f, (list, tuple)) else f.owner
    s = f[6] if isinstance(f, (list, tuple)) else f.ships
    fleet_ships_by_owner[o] += s

label = {-1: 'neutral', 0: 'P0', 1: 'P1'}
print(f"{'owner':10s} {'planets':>8s} {'planet_ships':>13s} {'fleet_ships':>12s} {'total_prod':>11s}")
for o in sorted(owner_count.keys()):
    print(f"{label.get(o, str(o)):10s} {owner_count[o]:8d} {ships_by_owner[o]:13d} "
          f"{fleet_ships_by_owner.get(o, 0):12d} {prod_by_owner[o]:11d}")

## 4. 인터랙티브 게임 재생

Kaggle 의 orbit_wars JS 비주얼라이저. 셀 출력에 player UI 가 뜸 — 재생/스텝/속도 조정.
노트북 외부 (예: `nbconvert`) 에서는 보이지 않으니 Jupyter 에서 직접 실행 필요.

In [ ]:
env.render(mode='ipython', width=900, height=700)

## 5. (옵션) HTML 파일로 저장

별도 브라우저 / 공유용. 위 셀에서 보는 UI 와 동일.

In [ ]:
html = env.render(mode='html')
out_path = REPO / 'notebooks' / 'last_game.html'
out_path.write_text(html)
print('saved:', out_path)
print('open in browser:', f'file://{out_path}')

## 6. (옵션) 다른 모델 vs 현재 모델

`ORBIT_WEIGHTS` 한 쪽만 다른 .pt 로 지정해서 비교 — Kaggle agent 는 module-level singleton 으로 모델 로드라, 두 다른 모델을 동시에 쓰려면 별도 process 필요. 간단 비교는 random opponent 추천:

In [ ]:
# 예: 우리 모델 vs random opponent
# env2 = make('orbit_wars', debug=False)
# env2.run([agent, 'random'])
# print('vs random — p0 reward:', env2.state[0].reward)
# env2.render(mode='ipython', width=900, height=700)